# 1- arXiv cs.AI Retrieval

Downloads the most recent cs.AI papers (https://arxiv.org/list/cs.AI/recent) into `data/downloaded/`.

In [5]:
#%pip install -q arxiv

In [ ]:
import os
import csv
import urllib.request
import arxiv

# Where the PDFs + metadata are saved (relative to notebooks/)
OUT_DIR = "../data/downloaded"
META_PATH = os.path.join(OUT_DIR, "metadata.csv")
FIELDS = ["id", "title", "authors", "published", "categories", "url", "abstract"]

# API client
client = arxiv.Client(delay_seconds=3)


# ---- GET calls: functions that talk to arXiv ----
def get_recent_papers(n):
    """GET https://export.arxiv.org/api/query?search_query=cat:cs.AI&sortBy=submittedDate&sortOrder=descending&max_results=n
    Returns the n most recent cs.AI papers with their metadata (title, abstract, authors, ...)."""
    search = arxiv.Search(query="cat:cs.AI", max_results=n, sort_by=arxiv.SortCriterion.SubmittedDate)
    return list(client.results(search))


def get_pdf(pdf_url):
    """GET https://arxiv.org/pdf/<id>
    Returns the PDF file as bytes."""
    with urllib.request.urlopen(pdf_url) as response:
        return response.read()

In [ ]:
# ---- Local files: saving PDFs + metadata (no network) ----

def pdf_path(paper_id):
    """Where a paper's PDF lives on disk: data/downloaded/<arxiv_id>.pdf"""
    return os.path.join(OUT_DIR, f"{paper_id}.pdf")


def load_known_ids():
    """Create the folder + CSV if needed and return the ids already in the CSV."""
    os.makedirs(OUT_DIR, exist_ok=True)
    if not os.path.exists(META_PATH):
        with open(META_PATH, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(FIELDS)
    with open(META_PATH, encoding="utf-8") as f:
        return {row["id"] for row in csv.DictReader(f)}


def save_pdf(paper_id, pdf_bytes):
    """Write the PDF bytes to disk."""
    with open(pdf_path(paper_id), "wb") as f:
        f.write(pdf_bytes)


def save_metadata(paper):
    """Append the paper's details as a row in metadata.csv and return them as a dict."""
    row = {
        "id": paper.get_short_id(),
        "title": paper.title,
        "authors": ", ".join(a.name for a in paper.authors),
        "published": paper.published.date(),
        "categories": ", ".join(paper.categories),
        "url": paper.entry_id,
        "abstract": " ".join(paper.summary.split()),
    }
    with open(META_PATH, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writerow(row)
    return row


def download_latest(n=10):
    """The "button": get the n newest papers, download + record the new ones.
    Returns the list of newly added papers (empty if there was nothing new)."""
    known_ids = load_known_ids()
    new_papers = []
    for paper in get_recent_papers(n):
        paper_id = paper.get_short_id()
        if not os.path.exists(pdf_path(paper_id)):  # also fills in a missing PDF for a paper already in the CSV
            save_pdf(paper_id, get_pdf(paper.pdf_url))
        if paper_id not in known_ids:
            new_papers.append(save_metadata(paper))
    return new_papers

In [7]:
# Press the button: run this cell whenever you want the latest papers
new_papers = download_latest(10)